# 32 · E3 Pulse Width-Amplitude 2D Matrix · 真器件

> ⚠️ 真连 B1500，接 pFeFET 器件。前置：E1 (30) PASS。

**目的**：寻找正 MW 工作窗口——扫描写脉冲幅值（3~6V）× 脉宽（1μs~300μs），
判断铁电极化翻转 (FE) 能否在某个区域超过陷阱俘获 (trap)。

序列（每点独立）：Reset → ERS(+V, t_w) → delay 10μs → 3pt-Read
                   Reset → PGM(−V, t_w) → delay 10μs → 3pt-Read
输出：MW(|V|, t_w) 2D 热图（4×6 = 24 点）。

测试人：**yhzang**

In [ ]:
import sys, os, time, random, datetime
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path, autodetect_visa_addr, clear_b1500_status_for_wgfmu_open,
    autodetect_wgfmu_chan, RealWgfmuBackend,
)

print("Python:", sys.version.split()[0])
print("project root:", ROOT)

In [ ]:
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

backend = RealWgfmuBackend()
backend.load()
idn = clear_b1500_status_for_wgfmu_open(VISA_ADDR)
print(f"B1500 preflight *CLS OK: {idn}")
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)
channel_ids = backend.get_channel_ids()
print(f"detected channels: {channel_ids}")

GATE_CH  = 202   # hardcoded: Gate=CH202 per verified yhzang wiring
DRAIN_CH = 201   # hardcoded: Drain=CH201 per verified yhzang wiring
assert GATE_CH in channel_ids and DRAIN_CH in channel_ids, f"gate/drain not found in {channel_ids}"
print(f"✅ gate={GATE_CH}, drain={DRAIN_CH}")
backend.close_session()

In [ ]:
# ── USER-EDITABLE PARAMETERS ──────────────────────────────────
QUICK_MODE  = True        # True: 2 amps × 3 widths × 2 reps; False: 4×6×5

DEVICE_ID   = "dev001"
GEOMETRY    = "W5L10"

AMP_QUICK   = [4.0, 5.0]           # |V| amplitudes in quick mode
AMP_FULL    = [3.0, 4.0, 5.0, 6.0]

WIDTH_QUICK = [1e-6, 10e-6, 100e-6]         # s, write pulse widths
WIDTH_FULL  = [1e-6, 3e-6, 10e-6, 30e-6, 100e-6, 300e-6]

T_RISE_FALL = 100e-9      # s
T_RESET     = 1e-3        # s
T_DELAY     = 10e-6       # s  fixed delay between write and read

VG_READS    = [-0.2, 0.0, +0.2]
VD_READ     = 0.05
T_READ      = 5e-6
N_PTS_READ  = 5

N_REPS_QUICK = 2
N_REPS_FULL  = 5
RANDOM_SEED  = 42
# ── END PARAMETERS ────────────────────────────────────────────

amps   = AMP_QUICK   if QUICK_MODE else AMP_FULL
widths = WIDTH_QUICK if QUICK_MODE else WIDTH_FULL
N_REPS = N_REPS_QUICK if QUICK_MODE else N_REPS_FULL
TAG    = "quick" if QUICK_MODE else "full"

import itertools
matrix_pts = list(itertools.product(amps, widths))
total_shots = len(matrix_pts) * N_REPS * 2  # 2 states per matrix point

print(f"Mode  : {TAG}")
print(f"Matrix: {len(amps)} amps × {len(widths)} widths = {len(matrix_pts)} points")
print(f"Total shots: {total_shots} = {len(matrix_pts)} × {N_REPS} reps × 2 states")

In [ ]:
def _run_shot(
    backend, gate_ch, drain_ch,
    v_write, t_write, t_rise_fall, t_reset, t_delay,
    vg_reads, vd_read, t_read, n_pts=5,
) -> list:
    """Single write-delay-read shot. Returns list of per-Vg_read dicts."""
    T_GAP  = 100e-9
    GUARD  = 200e-9
    t_delay = max(t_delay, 200e-9)

    backend.clear()

    backend.create_pattern("gp", 0.0)
    backend.add_vector("gp", t_reset,     0.0)
    backend.add_vector("gp", t_rise_fall, v_write)
    backend.add_vector("gp", t_write,     v_write)
    backend.add_vector("gp", t_rise_fall, 0.0)
    backend.add_vector("gp", t_delay,     0.0)

    t_pre = t_reset + t_rise_fall + t_write + t_rise_fall + t_delay
    backend.create_pattern("dp", 0.0)
    backend.add_vector("dp", t_pre, 0.0)

    meas_win = t_read - GUARD
    interval = meas_win / max(n_pts, 1)
    average  = interval * 0.8

    t_cursor = t_pre
    read_t0s = []
    for i, vg in enumerate(vg_reads):
        t_cursor += t_rise_fall
        read_t0s.append(t_cursor)
        t_cursor += t_read + t_rise_fall
        backend.add_vector("gp", t_rise_fall, float(vg))
        backend.add_vector("gp", t_read,       float(vg))
        backend.add_vector("gp", t_rise_fall,  0.0)
        backend.add_vector("dp", t_rise_fall, vd_read)
        backend.add_vector("dp", t_read,       vd_read)
        backend.add_vector("dp", t_rise_fall,  0.0)
        if i < len(vg_reads) - 1:
            t_cursor += T_GAP
            backend.add_vector("gp", T_GAP, 0.0)
            backend.add_vector("dp", T_GAP, 0.0)

    for i in range(len(vg_reads)):
        t_ev = read_t0s[i] + GUARD
        backend.set_measure_event("gp", f"g{i}", t_ev, n_pts, interval, average, "averaged")
        backend.set_measure_event("dp", f"d{i}", t_ev, n_pts, interval, average, "averaged")

    backend.add_sequence(gate_ch,  "gp", 1)
    backend.add_sequence(drain_ch, "dp", 1)

    backend.initialize()
    for ch in [gate_ch, drain_ch]:
        backend.set_operation_mode(ch, "FASTIV")
        backend.set_force_voltage_range(ch, "AUTO")
        backend.set_measure_enabled(ch, True)
        backend.set_measure_mode(ch, "CURRENT")
        backend.set_measure_current_range(ch, "1MA")
    backend.connect(gate_ch)
    backend.connect(drain_ch)

    backend.execute()
    backend.wait_until_completed()

    g_df = backend.get_measure_values(gate_ch)
    d_df = backend.get_measure_values(drain_ch)
    backend.disconnect(gate_ch)
    backend.disconnect(drain_ch)

    g_t = g_df["time_s"].values;  g_v = g_df["value"].values
    d_t = d_df["time_s"].values;  d_v = d_df["value"].values

    results = []
    for i, vg in enumerate(vg_reads):
        t0 = read_t0s[i] + GUARD
        t1 = t0 + meas_win
        d_sub = d_v[(d_t >= t0) & (d_t <= t1)]
        g_sub = g_v[(g_t >= t0) & (g_t <= t1)]
        results.append(dict(
            Vg_read_V=float(vg), Vd_read_V=vd_read,
            Id_mean_A = float(np.nanmean(d_sub)) if len(d_sub) > 0 else float("nan"),
            Id_std_A  = float(np.nanstd(d_sub))  if len(d_sub) > 1 else float("nan"),
            Ig_mean_A = float(np.nanmean(g_sub)) if len(g_sub) > 0 else float("nan"),
            n_d=int(len(d_sub)), n_g=int(len(g_sub)),
        ))
    return results


print("✅ _run_shot() defined")

In [ ]:
ts_start = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir  = ROOT / "runs" / f"{ts_start}_E3_pulse_matrix_{TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
out_csv  = run_dir / "pulse_matrix_results.csv"
print(f"Output : {run_dir}")

random.seed(RANDOM_SEED)
pts_run = list(matrix_pts)
random.shuffle(pts_run)
print(f"Shuffled {len(pts_run)} matrix points")

In [ ]:
all_rows = []
seq_id   = 0

idn = clear_b1500_status_for_wgfmu_open(VISA_ADDR)
print(f"B1500 preflight *CLS OK: {idn}")
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)
t_exp0 = time.time()
try:
    print(f"Starting E3 Pulse Matrix: {total_shots} shots")

    for rep in range(N_REPS):
        for amp, t_write in pts_run:
            for state, v_write in [("ERS", +amp), ("PGM", -amp)]:
                ts_iso = datetime.datetime.now().isoformat(timespec="seconds")
                t_sh   = time.time()
                note   = ""
                try:
                    rr = _run_shot(
                        backend, GATE_CH, DRAIN_CH,
                        v_write=v_write, t_write=t_write,
                        t_rise_fall=T_RISE_FALL, t_reset=T_RESET,
                        t_delay=T_DELAY, vg_reads=VG_READS,
                        vd_read=VD_READ, t_read=T_READ, n_pts=N_PTS_READ,
                    )
                except Exception as exc:
                    note = f"ERR:{exc}"
                    print(f"  !! {state} amp={amp}V tw={t_write:.1e}: {exc}")
                    rr = [dict(Vg_read_V=vg, Vd_read_V=VD_READ,
                               Id_mean_A=float("nan"), Id_std_A=float("nan"),
                               Ig_mean_A=float("nan"), n_d=0, n_g=0)
                          for vg in VG_READS]
                    for ch in [GATE_CH, DRAIN_CH]:
                        try: backend.disconnect(ch)
                        except Exception: pass
                    try: backend.close_session()
                    except Exception: pass
                    time.sleep(1.0)
                    idn = clear_b1500_status_for_wgfmu_open(VISA_ADDR)
                    print(f"B1500 preflight *CLS OK: {idn}")
                    backend.open_session(VISA_ADDR)
                    backend.set_timeout(30.0)

                for r in rr:
                    all_rows.append(dict(
                        timestamp_iso=ts_iso, device_id=DEVICE_ID,
                        geometry=GEOMETRY, sequence_id=seq_id,
                        repeat_index=rep, state_target=state,
                        v_write_V=v_write, amp_V=amp, t_write_s=t_write,
                        delay_s=T_DELAY,
                        Vg_read_V=r["Vg_read_V"], Vd_read_V=r["Vd_read_V"],
                        Id_mean_A=r["Id_mean_A"], Id_std_A=r["Id_std_A"],
                        Ig_mean_A=r["Ig_mean_A"], note=note,
                    ))
                seq_id += 1

            Id0e = all_rows[-3]["Id_mean_A"]  # ERS Vg=0V
            Id0p = all_rows[-len(VG_READS)]["Id_mean_A"]  # PGM Vg=0V (middle)
            print(f"  rep={rep} amp={amp}V tw={t_write:.1e} | "
                  f"Id_ERS={Id0e*1e9:.1f} Id_PGM={Id0p*1e9:.1f} nA | "
                  f"{time.time()-t_sh:.2f}s")

        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"\n✅ Done in {time.time()-t_exp0:.1f}s. Saved: {out_csv}")

except Exception as exc:
    print(f"\n❌ Failed: {exc}")
    raise
finally:
    for ch in [GATE_CH, DRAIN_CH]:
        try: backend.disconnect(ch)
        except Exception: pass
    backend.close_session()

df = pd.DataFrame(all_rows)
print(df.shape)

In [ ]:
df = pd.read_csv(out_csv)
vg_ref = 0.0
sub    = df[df["Vg_read_V"] == vg_ref]

amps_u  = sorted(sub["amp_V"].unique())
widths_u = sorted(sub["t_write_s"].unique())

mw_grid = np.full((len(amps_u), len(widths_u)), float("nan"))
for i, a in enumerate(amps_u):
    for j, w in enumerate(widths_u):
        s = sub[sub["amp_V"] == a][sub["t_write_s"] == w]
        id_ers = s[s["state_target"] == "ERS"]["Id_mean_A"].mean()
        id_pgm = s[s["state_target"] == "PGM"]["Id_mean_A"].mean()
        mw_grid[i, j] = (id_ers - id_pgm) * 1e9  # nA

vmax = max(abs(np.nanmin(mw_grid)), abs(np.nanmax(mw_grid)))
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(mw_grid, aspect="auto", cmap="RdBu",
               vmin=-vmax, vmax=vmax, origin="lower")
ax.set_xticks(range(len(widths_u)))
ax.set_xticklabels([f"{w*1e6:.0f}μs" for w in widths_u])
ax.set_yticks(range(len(amps_u)))
ax.set_yticklabels([f"{a:.0f}V" for a in amps_u])
ax.set_xlabel("Pulse width")
ax.set_ylabel("|V_write|")
ax.set_title(f"E3 MW = Id(ERS)−Id(PGM) (nA) @ Vg={vg_ref}V | {DEVICE_ID} {GEOMETRY}")
plt.colorbar(im, ax=ax, label="MW (nA)")
for i in range(len(amps_u)):
    for j in range(len(widths_u)):
        v = mw_grid[i, j]
        ax.text(j, i, f"{v:.0f}" if not np.isnan(v) else "?",
                ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(run_dir / "pulse_matrix_mw_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

pos = int(np.sum(mw_grid > 0))
print(f"MW > 0 at {pos}/{mw_grid.size} matrix points")
print(f"Max MW = {np.nanmax(mw_grid):.1f} nA,  Min MW = {np.nanmin(mw_grid):.1f} nA")

## 通过标准

- [ ] 无硬件错误，热图完整（24 or 6 点）
- [ ] 若 MW 矩阵全为负 → FE 操作窗口不存在 → 进 E4/E5 定位根因
- [ ] 若某区域 MW > 0 → 找到 FE 主导窗口 → 记录该 (amp, width) 组合

**结论记录**：MW > 0 出现于 (amp=___V, width=___μs)，最大 MW = ___ nA